# Análise Exploratória de Dados - Base Varejo

## Sprint 1: Importação e Reconhecimento da Base

Nesta etapa inicial, realizei a importação do arquivo `Base Varejo.csv` na plataforma Kaggle: https://www.kaggle.com/datasets/namespaiva/base-varejo/data
Durante o processo, identifiquei que o arquivo utiliza o ponto e vírgula (`;`) como separador de colunas.

Ao rodar o método `df.info()`, é possível perceber que a base conta com 830.000 registros.
Pontos de atenção nas próximas etapas:

- A coluna **DATA** foi importada como texto e precisará ser convertida para o formato correto de data.
- As colunas estão unidas por strings complexas em alguns pontos que avaliaremos na etapa de transformação.


In [1]:
# ==========================================
# SPRINT 1: IMPORTAÇÃO E EXPLORAÇÃO INICIAL
# ==========================================

import pandas as pd
import numpy as np

# Carregando a base de dados com o separador correto (ponto e vírgula)
df = pd.read_csv("Base Varejo.csv", sep=";")

# Visualizando as primeiras linhas para entender a estrutura dos dados
print("Primeiras linhas do dataset:")
display(df.head())

# Verificando informações gerais: quantidade de registros, colunas e tipos de dados
print("\nInformações gerais sobre a base de dados:")
df.info()


Primeiras linhas do dataset:


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,NaN,NaN,NaN,NaN
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,NaN,NaN,NaN,NaN
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,NaN,NaN,NaN,NaN
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO,NaN,NaN,NaN,NaN



Informações gerais sobre a base de dados:
<class 'pandas.DataFrame'>
RangeIndex: 830000 entries, 0 to 829999
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   DATA         830000 non-null  str    
 1   CO_ID        830000 non-null  int64  
 2   CL_ID        830000 non-null  int64  
 3   CL_GENERO    830000 non-null  str    
 4   CL_EC        830000 non-null  int64  
 5   CL_FHL       830000 non-null  int64  
 6   CL_SEG       830000 non-null  str    
 7   PR_ID        830000 non-null  int64  
 8   PR_CAT       830000 non-null  str    
 9   PR_NOME      830000 non-null  str    
 10  Unnamed: 10  0 non-null       float64
 11  Unnamed: 11  0 non-null       float64
 12  Unnamed: 12  0 non-null       float64
 13  Unnamed: 13  0 non-null       float64
dtypes: float64(4), int64(5), str(5)
memory usage: 88.7 MB


## Sprints 2 e 3: Transformação e Limpeza de Dados com Pandas

Com base nas metodologias de qualidade de dados estudadas em aula, realizei os seguintes tratamentos para assegurar a consistência da base de varejo antes das agregações analíticas:

* **Tratamento de Nulos das Dimensões Físicas:** Eliminação por completo as colunas residuais `Unnamed` utilizando o método `.drop()`. A escolha por essa exclusão é justificada pelo fato de esses atributos apresentarem 100% de omissão de dados (dados ausentes por falha de exportação), atuando apenas como ruído estrutural.
* **Normalização Textual:** Aplicação estruturada das funções `.str.strip()` e `.str.upper()` nas colunas de produto para mitigar problemas de espaçamento e duplicidade por variação de caixa.
* **Tratamento de Omissões por Lógica Condicional:** Desenvolvimento de um laço iterativo baseado na estrutura condicional `if/else`. Essa regra de negócio localiza campos de categoria vazios ou nulos e faz a imputação pelo rótulo padronizado "Sem Categoria".
* **Conversão de Data via Datetime:** Ajuste e normalização da coluna `DATA` convertendo o formato de texto original para o tipo temporal `datetime64`, utilizando o parâmetro de coerção para isolar inconsistências.
* **Validação do Identificador de Compra (CO_ID):** Execução do método `.drop_duplicates()` para eliminar registros redundantes exatos, garantindo a integridade dos dados e a validação lógica das transações comerciais.


In [4]:
# =====================================================================
# SPRINT 2 & 3: TRANSFORMAÇÃO DE TIPOS E TRATAMENTO DE NULOS
# =====================================================================

# Tratamento de nulos das dimensões físicas
# Justificativa técnica: As colunas extras 'Unnamed' apresentam 100% de valores ausentes (nulos).
# Optou-se pela exclusão de colunas (Trimming), pois não possuem relevância para as regras de negócio.
colunas_vazias = ['Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']
df = df.drop(columns=colunas_vazias, errors='ignore')  # errors='ignore' evita erros caso as colunas não existam

# SPRINT 2: Padronização de Texto
df['PR_CAT'] = df['PR_CAT'].str.strip().str.upper()
df['PR_NOME'] = df['PR_NOME'].str.strip().str.upper()

# Lógica condicional (if/else) para preencher categorias vazias
# Garante que campos em branco ou strings ruidosas sejam mapeados uniformemente
categorias_ajustadas = []
for item in df['PR_CAT']:
    if item == "" or item == "NAN" or item == "NONE":
        categorias_ajustadas.append("Sem Categoria")
    else:
        categorias_ajustadas.append(item)
df['PR_CAT'] = categorias_ajustadas

# Conversão de string de data utilizando o módulo datetime
# O parâmetro errors='coerce' é empregado para mitigar eventuais inconsistências de datas
df['DATA'] = pd.to_datetime(df['DATA'], format='%d/%m/%Y', errors='coerce')

# Validação da regra do identificador de número de compra (CO_ID)
# Remoção de duplicatas idênticas para preservar a unicidade de cada transação estruturada
df = df.drop_duplicates()

print("--- DIAGNÓSTICO DA BASE LIMPA ---")
df.info()


--- DIAGNÓSTICO DA BASE LIMPA ---
<class 'pandas.DataFrame'>
Index: 733447 entries, 0 to 829999
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   DATA       733447 non-null  datetime64[us]
 1   CO_ID      733447 non-null  int64         
 2   CL_ID      733447 non-null  int64         
 3   CL_GENERO  733447 non-null  str           
 4   CL_EC      733447 non-null  int64         
 5   CL_FHL     733447 non-null  int64         
 6   CL_SEG     733447 non-null  str           
 7   PR_ID      733447 non-null  int64         
 8   PR_CAT     733447 non-null  str           
 9   PR_NOME    733447 non-null  str           
dtypes: datetime64[us](1), int64(5), str(4)
memory usage: 61.6 MB
